In [ ]:
import os
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, Word2Vec
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [ ]:
DATA_PATH = os.getenv("REVIEWS_DATA_PATH", "../data/raw/reviews.jsonl")

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("AmazonReviews").getOrCreate()

In [ ]:
df = spark.read.json(DATA_PATH)
df.show()
df.printSchema()

In [ ]:
positive_reviews = df.select("text").where("label = 1")

In [ ]:
positive_reviews.show()

In [ ]:
negative_reviews = df.select("text").where("label = 0")

In [ ]:

pos_df = positive_reviews.select("text").withColumn("label", F.lit(1))
neg_df = negative_reviews.select("text").withColumn("label", F.lit(0))


reviews_df = pos_df.unionByName(neg_df)

reviews_df = reviews_df.withColumn(
    "text",
    F.lower(F.regexp_replace(F.col("text"), r"[^a-zA-Z\s]", ""))
)

reviews_df = reviews_df.filter(F.length(F.trim(F.col("text"))) > 0)

train_df, test_df = reviews_df.randomSplit([0.8, 0.2], seed=42)

tokenizer = RegexTokenizer(
    inputCol="text",
    outputCol="tokens",
    pattern="\\s+"
)

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

word2vec = Word2Vec(
    inputCol="filtered_tokens",
    outputCol="features",
    vectorSize=100,
    minCount=2
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20
)

pipeline = Pipeline(stages=[tokenizer, remover, word2vec, lr])


model = pipeline.fit(train_df)

predictions = model.transform(test_df)


predictions.select("text", "label", "probability", "prediction").show(20, truncate=False)


binary_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

acc_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

auc = binary_eval.evaluate(predictions)
f1 = f1_eval.evaluate(predictions)
accuracy = acc_eval.evaluate(predictions)

print("AUC:", auc)
print("F1:", f1)
print("Accuracy:", accuracy)

## Getting 80% accuracy on our first run

## Trying a different approach below

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression

tokenizer = RegexTokenizer(inputCol="text", outputCol="tokens", pattern="\\s+")
remover = StopWordsRemover(inputCol="tokens", outputCol="filtered_tokens")
tf = HashingTF(inputCol="filtered_tokens", outputCol="rawFeatures", numFeatures=20000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)

tfidf_pipeline = Pipeline(stages=[tokenizer, remover, tf, idf, lr])
tfidf_model = tfidf_pipeline.fit(train_df)
tfidf_predictions = tfidf_model.transform(test_df)

In [ ]:
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = auc_evaluator.evaluate(tfidf_predictions)
print("AUC:", auc)

In [ ]:
spark.stop()